In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from pathlib import Path

load_dotenv()

client = Anthropic(
    default_headers={
        "anthropic-beta": "code-execution-2025-08-25, files-api-2025-04-14"
    }
)
model = "claude-sonnet-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=2000,
):
    params = {
        "model": model,
        "max_tokens": 10000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


def upload(file_path):
    path = Path(file_path)
    extension = path.suffix.lower()

    mime_type_map = {
        ".pdf": "application/pdf",
        ".txt": "text/plain",
        ".md": "text/plain",
        ".py": "text/plain",
        ".js": "text/plain",
        ".html": "text/plain",
        ".css": "text/plain",
        ".csv": "text/csv",
        ".json": "application/json",
        ".xml": "application/xml",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        ".xls": "application/vnd.ms-excel",
        ".jpeg": "image/jpeg",
        ".jpg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }

    mime_type = mime_type_map.get(extension)

    if not mime_type:
        raise ValueError(f"Unknown mimetype for extension: {extension}")
    filename = path.name

    with open(file_path, "rb") as file:
        return client.beta.files.upload(file=(filename, file, mime_type))


def list_files():
    return client.beta.files.list()


def delete_file(id):
    return client.beta.files.delete(id)


def download_file(id, filename=None):
    file_content = client.beta.files.download(id)

    if not filename:
        file_metadata = get_metadata(id)
        file_content.write_to_file(file_metadata.filename)
    else:
        file_content.write_to_file(filename)


def get_metadata(id):
    return client.beta.files.retrieve_metadata(id)

In [3]:
file_metadata = upload("streaming.csv")
file_metadata

BetaFileMetadata(id='file_01JSXYUF823uF8WXhZHaNjmb', created_at=datetime.datetime(2026, 8, 26, 12, 7, 4, 59241, tzinfo=datetime.timezone.utc), filename='streaming.csv', mime_type='text/csv', size_bytes=25733, type='file', downloadable=False, scope=None)

In [4]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": """
Run a detailed analysis to determine major drivers of churn.
Your final output should include at least one detailed plot summarizing your findings.

Critical note: Every time you execute code, you're starting with a completely clean slate. 
No variables or library imports from previous executions exist. You need to redeclare/reimport all variables/libraries.
            """,
        },
        {"type": "container_upload", "file_id": file_metadata.id},
    ],
)

chat(messages, tools=[{"type": "code_execution_20250825", "name": "code_execution"}])

Message(id='msg_011CeRLFKogjMRAJo9hWMQWp', container=Container(id='container_01EnX1SXKGYvX7Ymfs4tnaht', expires_at=datetime.datetime(2026, 8, 26, 13, 10, 37, 531904, tzinfo=TzInfo(0)), skills=None), content=[ThinkingBlock(signature='EpACCpABCBEYAipAUE4c3o/pkoMJaI9LbukhUIKT84SeHTAGaDg+6N/hh2c2RSbRkdW8itKzY6nhrwmILP8pKkT6VobXmNEZjTII8DIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiQwZGRiMTdjOS00N2FmLTQ3YzAtOWE0NS05OTAzYWRkNTc4MWOoAYuuu9QGEgwx3x78MbImRoTGSHgaDE4IJFAHZa0HaL4q8yIwu8vkC1z4jpjhkcRbDsGQ+IaPPbdDWxMjMIsa2cW9C+i5ht0KhghDQMj0sGaGP9ppKi3sL/JvRUzgdUYS0xFPCni1TYhiolg6UYJVW7zEh27IMMndz7218jcdz4Vejf0YAQ==', thinking='', type='thinking'), ServerToolUseBlock(id='srvtoolu_012Ssyf29ktF87ofekx2kyKq', caller=None, input={'command': 'cd $INPUT_DIR && ls -la && echo "---" && head -5 streaming.csv && echo "---" && wc -l streaming.csv'}, name='bash_code_execution', type='server_tool_use'), BashCodeExecutionToolResultBlock(content=BashCodeExecutionResultBlock(content=[], return_code=0, stderr='', stdout

In [5]:
download_file('file_01GWLtDdW886YkAWXeihFZHb')